# Stage 4.5 v4 - TrackNet ball detector (Run 2: all-venue, warm-started)

**Upload this notebook, Run All.** Mixed precision (AMP) on; `BATCH` auto-fits the GPU.

**Needs in `MyDrive/`:**
- `pb_v4_upload.zip` - the data+code bundle (all 6 clips' caches). Already current.
- `ball_model_v4_base.pt` - the clean same-court v4 baseline (recall 0.90 @ fp 0.018)
  to WARM-START from. Upload your local `data/models/ball_model_v4.pt` under this name.

**Run 2 (production, all 3 venues):** trains home outdoor (pb_3/4/5min) + court2
outdoor + indoor. Measurement uses PER-VENUE HELD-OUT SLICES (leakage-free): pb_2min
held out entirely as the same-court guardrail; court2 and indoor each split 88% train
/ 12% held-out val. Venue-balanced sampling + photometric augmentation are the
cross-venue levers.

**Warm-start + selection:** initializes from the 0.90 baseline (preserves same-court
precision) and fine-tunes at `LR 1e-4`. Model selection = highest MEAN per-venue recall
among epochs whose HOME fp stays <= 0.05. Best saved to `ball_model_v4_run2.pt`
(+ `validation_report_run2.json`) - the baseline is never overwritten.

**Runtime -> Change runtime type -> GPU.** Lower `BATCH` in KNOBS if OOM.

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # before torch/CUDA

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import zipfile, sys
from pathlib import Path
LOCAL = Path('/content/pb_v4')
if not LOCAL.exists():
    print('unzipping bundle to local disk...')
    with zipfile.ZipFile('/content/drive/MyDrive/pb_v4_upload.zip') as z:
        z.extractall(LOCAL)
REPO = LOCAL/'repo'; DATA = LOCAL/'data'
sys.path.insert(0, str(REPO))
print('clips:', sorted(p.name for p in DATA.iterdir()))

In [ ]:
try:
    import cv2
except Exception:
    !pip -q install opencv-python-headless
    import cv2

In [ ]:
# ===== KNOBS (edit if needed) =====
import torch as _t
if _t.cuda.is_available():
    _gb = _t.cuda.get_device_properties(0).total_memory / 1e9
    BATCH = 2 if _gb < 18 else (4 if _gb < 34 else (8 if _gb < 50 else 12))  # T4->2 L4/A10->4 A100-40->8 A100-80->12
    print(f'GPU {_gb:.0f} GB -> auto BATCH {BATCH}')
else:
    BATCH = 2
EPOCHS = 15
LR     = 1e-4     # warm-start FINE-TUNE rate (lower than the 5e-4 from-scratch rate) to preserve same-court precision
ACCUM  = 4        # grad accumulation -> effective batch = BATCH*ACCUM (stability, no extra mem)
PROC_H, PROC_W = 720, 1280
TOL, CONF, SIGMA = 6, 0.30, 3.0

import json, numpy as np, torch, random; cv2.setNumThreads(0)
torch.manual_seed(0); np.random.seed(0); random.seed(0)  # reproducible
from torch.utils.data import Dataset, DataLoader
from stages.track_ball._tracknet_model import TrackNet   # the only bundle import (stable)

def make_heatmap(x, y):
    hm = np.zeros((PROC_H, PROC_W), np.float32)
    if x is None or y is None: return hm
    ix, iy = int(round(x)), int(round(y))
    if not (0 <= ix < PROC_W and 0 <= iy < PROC_H): return hm
    r = int(3*SIGMA)
    x0, x1 = max(0, ix-r), min(PROC_W, ix+r+1)
    y0, y1 = max(0, iy-r), min(PROC_H, iy+r+1)
    xs = np.arange(x0, x1)[None, :]; ys = np.arange(y0, y1)[:, None]
    hm[y0:y1, x0:x1] = np.exp(-((xs-ix)**2 + (ys-iy)**2)/(2*SIGMA**2)).astype(np.float32)
    hm[iy, ix] = 1.0
    return hm

def focal(pred, gt, a=2.0, b=4.0, eps=1e-6):
    pred = pred.clamp(eps, 1-eps)
    pos = gt.eq(1.0).float(); neg = 1 - pos
    pl = ((1-pred)**a) * torch.log(pred) * pos
    nl = ((1-gt)**b) * (pred**a) * torch.log(1-pred) * neg
    return -(pl.sum() + nl.sum()) / pos.sum().clamp(min=1.0)

class DS(Dataset):
    def __init__(self, items, aug=False):   # items: list of (sample_dict, frames_dir Path)
        self.items = list(items); self.aug = aug
    def __len__(self): return len(self.items)
    def rd(self, fd, i):
        im = cv2.imread(str(fd/f'{i}.jpg'))
        if im is None: return None
        if im.shape[:2] != (PROC_H, PROC_W): im = cv2.resize(im, (PROC_W, PROC_H))
        return cv2.cvtColor(im, cv2.COLOR_BGR2RGB).astype(np.float32)/255.0
    def __getitem__(self, k):
        s, fd = self.items[k]
        frs = [self.rd(fd, i) for i in s['frames']]
        if any(f is None for f in frs):
            return (torch.zeros(9, PROC_H, PROC_W), torch.zeros(1, PROC_H, PROC_W), -1.0, -1.0)
        st = np.concatenate([f.transpose(2, 0, 1) for f in frs], 0).astype(np.float32)
        if s['visible']:
            tg = make_heatmap(s['x_proc'], s['y_proc'])[None]
            px, py = float(s['x_proc']), float(s['y_proc'])
        else:
            tg = np.zeros((1, PROC_H, PROC_W), np.float32); px = py = -1.0
        if self.aug:
            # horizontal flip - court orientation is irrelevant to a ball detector
            if np.random.rand() < 0.5:
                st = st[:, :, ::-1].copy(); tg = tg[:, :, ::-1].copy()
                if px >= 0: px = PROC_W - 1 - px
            # photometric - the key cross-venue lever (indoor vs outdoor lighting +
            # white balance). Same transform across all 3 stacked frames (9ch).
            if np.random.rand() < 0.7:
                cg = (1 + (np.random.rand(3)-0.5)*0.25).astype(np.float32)  # per-RGB gain 0.75-1.25 (color temp)
                gain = np.tile(cg, 3)[:, None, None]
                bs = np.float32((np.random.rand()-0.5)*0.2)                # brightness +/-0.1
                st = np.clip(st*gain + bs, 0, 1).astype(np.float32)
            if np.random.rand() < 0.5:
                m = np.float32(st.mean()); c = np.float32(1 + (np.random.rand()-0.5)*0.2)  # contrast +/-25%
                st = np.clip((st-m)*c + m, 0, 1).astype(np.float32)
        return torch.from_numpy(st), torch.from_numpy(tg.astype(np.float32)), px, py

def clip_items(folder):
    """All (sample, frames_dir) items for one clip, read from its v4_manifest.json."""
    m = json.load(open(folder/'v4_manifest.json'))
    fd = folder/m.get('frames_dir', 'frames_720')
    return [(s, fd) for s in m['samples']]

def _frame_key(s):
    fr = s['frames']
    return fr[len(fr)//2] if isinstance(fr, (list, tuple)) else fr

def temporal_split(items, val_frac=0.12):
    """Hold out the LAST val_frac of a clip by frame time as a contiguous block, so
    the val slice shares (almost) no near-duplicate adjacent frames with train."""
    order = sorted(range(len(items)), key=lambda k: _frame_key(items[k][0]))
    n_val = max(1, int(round(len(items) * val_frac)))
    val_pos = set(order[-n_val:])
    tr = [items[k] for k in range(len(items)) if k not in val_pos]
    va = [items[k] for k in order[-n_val:]]
    return tr, va

def peak(hm):
    iy, ix = np.unravel_index(int(hm.argmax()), hm.shape)
    return ix, iy, float(hm[iy, ix])

@torch.no_grad()
def evaluate(model, ds, dev):
    model.eval(); dl = DataLoader(ds, batch_size=BATCH, num_workers=8)
    tp = fn = fp = tn = 0
    for st, tg, px, py in dl:
        with torch.cuda.amp.autocast(enabled=dev == 'cuda'):
            out = model(st.to(dev))
        pr = out.float().cpu().numpy()
        for j in range(pr.shape[0]):
            x, y, v = peak(pr[j, 0]); det = v >= CONF
            gx, gy = float(px[j]), float(py[j])
            if gx >= 0:
                ok = det and (np.hypot(x-gx, y-gy) <= TOL)
                tp += int(ok); fn += int(not ok)
            else:
                fp += int(det); tn += int(not det)
    return tp/max(tp+fn, 1), fp/max(fp+tn, 1)

print('training utils defined; BATCH =', BATCH)


In [ ]:
import os
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---- Run 2: production model on ALL 3 venues, with per-venue HELD-OUT slices ----
# home outdoor: pb_2min held out entirely as the same-court guardrail (comparable to
# the 0.90 baseline); pb_3/4/5min train. court2 + indoor each split 88% train /
# 12% held-out val, so every venue has a leakage-free recall readout.
HOME_TRAIN  = ['pb_3min', 'pb_4min', 'pb_5min']   # home clips used for training
HOME_VAL    = 'pb_2min'                            # home clip held out entirely (same-court guardrail)
SLICE_CLIPS = {'court2': 'pb_3min_court2', 'indoor': 'pb_3min_indoor'}
VAL_FRAC    = 0.12

train_items = []
for c in HOME_TRAIN:
    train_items += clip_items(DATA/c)
val_sets = {'home': DS(clip_items(DATA/HOME_VAL))}
for ven, clip in SLICE_CLIPS.items():
    tr, va = temporal_split(clip_items(DATA/clip), VAL_FRAC)
    train_items += tr
    val_sets[ven] = DS(va)
train = DS(train_items, aug=True)
print('train', len(train), '| val:', {k: len(v) for k, v in val_sets.items()})

# Venue-balanced sampling: each venue contributes equally per epoch (weight each
# training sample by 1 / its venue's train count).
CLIP_VENUE = {'pb_2min':'home','pb_3min':'home','pb_4min':'home','pb_5min':'home',
              'pb_3min_court2':'court2','pb_3min_indoor':'indoor'}
_ven = [CLIP_VENUE.get(fd.parent.name, 'home') for (s, fd) in train.items]
from collections import Counter
_vc = Counter(_ven); print('train venue mix:', dict(_vc))
_w = torch.tensor([1.0/_vc[v] for v in _ven], dtype=torch.double)
sampler = torch.utils.data.WeightedRandomSampler(_w, num_samples=len(train), replacement=True)

model = TrackNet(in_channels=9, out_channels=1, input_shape=(PROC_H, PROC_W)).to(dev)
# WARM-START from the clean same-court v4 baseline (recall 0.90 @ fp 0.018) so
# multi-venue training ADAPTS rather than relearning from scratch -> keeps same-court
# precision. Upload your local data/models/ball_model_v4.pt to MyDrive under this name.
BASE = '/content/drive/MyDrive/ball_model_v4_base.pt'
if os.path.exists(BASE):
    _sd = torch.load(BASE, map_location=dev)
    model.load_state_dict(_sd['state_dict'])
    print(f"warm-started from baseline (val_recall {_sd.get('val_recall')}, fp {_sd.get('val_fp')})")
else:
    print('WARNING: baseline not found -> training FROM SCRATCH:', BASE)
opt   = torch.optim.Adam(model.parameters(), lr=LR)
scaler = torch.cuda.amp.GradScaler(enabled=dev == 'cuda')

def _fits(bs):
    try:
        torch.cuda.empty_cache()
        st = torch.zeros(bs, 9, PROC_H, PROC_W, device=dev)
        tg = torch.zeros(bs, 1, PROC_H, PROC_W, device=dev)
        with torch.cuda.amp.autocast(enabled=dev == 'cuda'):
            out = model(st)
        focal(out.float(), tg).backward()
        model.zero_grad(set_to_none=True)
        del st, tg, out
        torch.cuda.empty_cache()
        return True
    except RuntimeError as e:
        model.zero_grad(set_to_none=True)
        torch.cuda.empty_cache()
        if 'out of memory' in str(e).lower():
            return False
        raise
if dev == 'cuda':
    while BATCH > 1 and not _fits(BATCH):
        print(f'  BATCH {BATCH} OOM -> trying {BATCH // 2}')
        BATCH = BATCH // 2
print(f'using BATCH {BATCH} x ACCUM {ACCUM} = effective {BATCH*ACCUM}')

sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
dl = DataLoader(train, batch_size=BATCH, sampler=sampler, num_workers=8)

# Model selection: maximize MEAN per-venue recall, but ONLY among epochs whose HOME
# false-positive rate stays within FP_CAP (guards same-court precision - the old
# `recall - fp` metric discarded the good-recall epochs). Saved to a NEW file so the
# 0.90 baseline is never overwritten; nothing is saved if home precision can't hold.
FP_CAP = 0.05
RUN2_CKPT   = '/content/drive/MyDrive/ball_model_v4_run2.pt'
RUN2_REPORT = '/content/drive/MyDrive/validation_report_run2.json'
best = -1.0
best_res = None
for ep in range(EPOCHS):
    model.train(); tot = 0.0; opt.zero_grad()
    for i, (st, tg, _, _) in enumerate(dl):
        with torch.cuda.amp.autocast(enabled=dev == 'cuda'):
            pred = model(st.to(dev))
        loss = focal(pred.float(), tg.to(dev).float()) / ACCUM
        scaler.scale(loss).backward()
        if (i + 1) % ACCUM == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler.step(opt); scaler.update(); opt.zero_grad()
        tot += loss.item() * ACCUM
    sched.step()
    res = {ven: evaluate(model, ds, dev) for ven, ds in val_sets.items()}  # {ven: (recall, fp)}
    hr, hfp = res['home']
    mean_rec = sum(r for r, _ in res.values()) / len(res)
    msg = (f'epoch {ep}: loss {tot/len(dl):.3f}  ' +
           '  '.join(f'{v}_rec {r:.3f}/fp {f:.3f}' for v, (r, f) in res.items()) +
           f'  mean_rec {mean_rec:.3f}')
    if hfp <= FP_CAP and mean_rec > best:
        best = mean_rec; best_res = res
        torch.save({'state_dict': model.state_dict(), 'input_shape': (PROC_H, PROC_W),
                    'in_channels': 9, 'out_channels': 1,
                    'per_venue': {v: {'recall': r, 'fp': f} for v, (r, f) in res.items()},
                    'mean_recall': mean_rec, 'warm_started': os.path.exists(BASE), 'run': 'run2'},
                   RUN2_CKPT)
        json.dump({'best_mean_recall': mean_rec, 'fp_cap': FP_CAP,
                   'per_venue': {v: {'recall': r, 'fp': f} for v, (r, f) in res.items()},
                   'home_val_clip': HOME_VAL, 'slice_clips': SLICE_CLIPS, 'val_frac': VAL_FRAC},
                  open(RUN2_REPORT, 'w'), indent=1)
        msg += '  <- saved'
    print(msg)
if best_res:
    print('DONE. best per-venue: ' + '  '.join(f'{v}_rec {r:.3f}/fp {f:.3f}' for v, (r, f) in best_res.items()))
else:
    print('DONE. no epoch met the home fp cap -> nothing saved (home precision could not be held).')


## Done
- **home_rec** (pb_2min, same-court guardrail) should hold >= 0.90 at fp <= 0.05.
- **court2_rec / indoor_rec** (held-out slices) - the cross-venue numbers that matter.
- Download `MyDrive/ball_model_v4_run2.pt` -> `data/models/` and compare
  `validation_report_run2.json` to the baseline before adopting it as production.